# P6 LLM 判定の偏りを測る

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

ソラハ電機（架空）の返信の下書きを 2 本ずつ比べさせ、判定役の位置・長さ・自分びいきの偏りを測ります。判定役は主に Sonnet 5.5、比較に Opus 5.5。

- 上から順に実行します。0 のセルは準備です（Colab でも手元の Jupyter でも同じ）。
- API キーが無ければ講師の記録の再生（replay）で、本物の実行と同じ数字が出ます（費用 0）。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る（Colab のときだけ取得とインストールをする）
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/llm-eval-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/llm-eval-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根（common/ がある所）を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/llm.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. AI の呼び先を決める（キーは表示しない）
from common import nb
from common.nb import choose_provider, safe, show, show_file, lab
PROVIDER = choose_provider(IN_COLAB)

## 画面に収める小さな道具
長い行を画面の幅で折り返して表示するだけの関数です（中身は変えません）。

In [ ]:
import io, contextlib
def fit(fn, *a, w=104, **k):   # fn の表示を、全角 2・半角 1 で数えた幅 w で折り返す
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        fn(*a, **k)
    for line in buf.getvalue().splitlines():
        out, n = "", 0
        for ch in line:
            d = 2 if ord(ch) > 0x2E80 else 1
            if n + d > w:
                print(out)
                out, n = "        ", 8
            out, n = out + ch, n + d
        print(out)

## 1. 入れ替えて 2 回聞くコード
`common/judge.py` の本物です。順番を入れ替えて結論が変わったら引き分けにします（Wang et al. 2023 の考え方）。

In [ ]:
from common import kit, llm as L, judge as J, stats
fit(show, J.pairwise)

## 2. 素朴な依頼文で、同じ質の組 EQ-13 を 2 回判定する
判定役は Sonnet 5.5（記録の再生）。中身が同じで言い回しだけ違う 2 本の返信です。

In [ ]:
P = "replay" if PROVIDER == "mock" else PROVIDER
naive = J.load_prompt("lab06_judge_bias/prompts/pairwise_naive.md")
x = next(p for p in kit.load_jsonl("pairs.jsonl") if p["id"] == "EQ-13")
with safe():
    v = J.pairwise(naive, f"受付日: {x['received']}\n{x['inquiry']}", x["a"], x["b"], provider=P, model="claude-sonnet-5-5")
    print("1 回目（返信 x が A・返信 y が B）→ 勝ち:", v["first_order"])
    print("2 回目（返信 y が A・返信 x が B）→ 勝ち:", v["swapped"])
    print("先に出た方が勝ったか:", v["first_pos_won"], "/ 入れ替えても同じ結論:", v["consistent"], "→ 最終:", v["final"])

## 3. 素朴な依頼文で 20 組 × 2 回 ― Sonnet と Opus
`run_bias.py` の本物を回し、位置の結果だけを短く表示します。

In [ ]:
import io, contextlib
rb = lab("run_bias", "lab06_judge_bias")
def bias(model, prompt, kinds):
    with contextlib.redirect_stdout(io.StringIO()):
        return rb.main(["--provider", P, "--model", model, "--prompt", prompt, "--kinds", kinds, "--all"])
for model in ("claude-sonnet-5-5", "claude-opus-5-5"):
    with safe():
        e = bias(model, "naive", "equal,length")["equal"]
        print(f"{model:<18} 入れ替えても同じ結論 {e['consistent']}/{e['pairs']}  最終の引き分け {e['final']['tie']}/{e['pairs']}"
              f"  先に出た方の勝ち {e['first_pos_wins']}/{e['decided']}")

## 4. 依頼文に基準を書く
素朴な依頼文（上）と、基準つきの依頼文（下）。`lab06_judge_bias/prompts/` の本物です。

In [ ]:
show_file("lab06_judge_bias/prompts/pairwise_naive.md", 1, 1)
print()
fit(show_file, "lab06_judge_bias/prompts/pairwise.md", 1, 15)

## 5. 基準つきの依頼文で、同じ質の 20 組を判定する

In [ ]:
for model, kinds in (("claude-sonnet-5-5", "equal,length,long_wrong"), ("claude-opus-5-5", "equal")):
    with safe():
        e = bias(model, "careful", kinds)["equal"]
        print(f"{model:<18} 入れ替えても同じ結論 {e['consistent']}/{e['pairs']}  最終の引き分け {e['final']['tie']}/{e['pairs']}"
              f"  先に出た方の勝ち {e['first_pos_wins']}/{e['decided']}")

## 6. 長さの偏り ― 中身が同じで長いだけの組、長いが誤っている組
教科書は「長い方をひいきする」と言いますが、向きは測るまでわかりません。

In [ ]:
for model in ("claude-sonnet-5-5", "claude-opus-5-5"):
    with safe():
        e = bias(model, "naive", "equal,length")["length"]
        print(f"素朴  {model:<18} 長い方を選んだ {e['per_order_y']}/{e['judgments']} 回"
              f"  最終: 長い方 {e['final']['y']} / 短い方 {e['final']['x']} / 引き分け {e['final']['tie']}")
with safe():
    r = bias("claude-sonnet-5-5", "careful", "equal,length,long_wrong")
    e, w = r["length"], r["long_wrong"]
    print(f"基準つき Sonnet 5.5   長い方を選んだ {e['per_order_y']}/{e['judgments']} 回"
          f"  最終: 長い方 {e['final']['y']} / 短い方 {e['final']['x']} / 引き分け {e['final']['tie']}")
    print(f"基準つき Sonnet 5.5   長いが誤り {w['pairs']} 組: 短く正しい方 {w['final']['x']} / 長く誤った方 {w['final']['y']}")

## 7. 自分びいき ― 書き手 2 × 判定役 2、10 件
Opus と Sonnet が同じ問い合わせに返信を書き、両方が判定役として比べます（入れ替えて 2 回ずつ）。

In [ ]:
rs = lab("run_selfpref", "lab06_judge_bias")
with safe():
    with contextlib.redirect_stdout(io.StringIO()):
        r = rs.main(["--provider", P, "--limit", "10"])
    for jm, s in r["judges"].items():
        print(f"判定役 {jm:<18} Opus の返信の勝ち {s['opus_wins']:>2} / Sonnet の返信の勝ち {s['sonnet_wins']:>2}"
              f" / 引き分け {s['ties']:>2} → Opus の得点 {s['opus_score']:.2f}")
    print(f"自分びいきの目安 {r['self_pref']:+.2f}  問いごとの差の 95% 区間 [{r['diff_ci'][0]:+.2f}, {r['diff_ci'][1]:+.2f}]（n={r['n']}）")
    print("返信の平均の長さ: " + " / ".join(f"{w} {v:.0f} 字" for w, v in r["avg_len"].items()) + f" / 費用 {r['cost_usd']} USD")

## 8. 例題 ― 入れ替え 2 回の結果表を読む
素朴な依頼文・Sonnet 5.5 の、同じ質の 20 組（上の 3 で書き出した本物の記録）。

In [ ]:
from collections import Counter
runs = [r for r in kit.read_jsonl("lab06_judge_bias/out/runs_bias_naive_sonnet-5-5.jsonl") if r["kind"] == "equal"]
tab = Counter((r["first_order"], r["swapped"]) for r in runs)
ks = ["x", "y", "tie"]
print(kit.table([[f"1 回目 {a}"] + [tab[(a, b)] for b in ks] for a in ks], ["", "2 回目 x", "2 回目 y", "2 回目 tie"]))
print(f"対角（入れ替えても同じ）= {' + '.join(str(tab[(k, k)]) for k in ks)} = {sum(tab[(k, k)] for k in ks)}/{len(runs)}")
dec = [fp for r in runs for v, fp in zip((r["first_order"], r["swapped"]), r["first_pos_won"]) if v != "tie"]
print(f"勝ち負けを付けた {len(dec)} 回のうち、先に出た方の勝ち {sum(dec)} 回 = {kit.pct(sum(dec) / len(dec))}"
      f" 95%区間 {kit.ci_str(*stats.wilson(sum(dec), len(dec)))}")
print("最終（対角はそのまま、対角の外は引き分け）:", dict(Counter(r["final"] for r in runs)))